In [1]:
#build best model

In [2]:
import tensorflow as tf
print(tf.__version__)

import sklearn
print(sklearn.__version__)

2.21.0
1.7.2


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

In [4]:
import pandas as pd  # Load the fake job postings dataset
df = pd.read_csv("/Users/nitesh/Downloads/fake_job_postings.csv", low_memory=False)

In [5]:
df = df.astype(object)                         # Convert all columns to object type
df.fillna("", inplace=True)                    # replace missing values with empty strings

In [6]:
df['salary_range'] = df['salary_range'].astype(str) # Convert salary_range column to string type

In [7]:
df['x'] = (
    df['title'] + " " +
    df['company_profile'] + " " +
    df['description'] + " " +
    df['requirements'] + " " +
    df['benefits'] + " " +
    df['salary_range']
)                                            # Combine multiple text columns into a single feature for model input
print(df.shape)
df.head()

(17880, 36)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,...,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,x
0,1,Marketing Intern,"US, NY, New York",Marketing,,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,...,,,,,,,,,Y,"Marketing Intern We're Food52, and we've creat..."
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,...,,,,,,,,,,Customer Service - Cloud Video Production 90 S...
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",,,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,0,...,,,,,,,,,,Commissioning Machinery Assistant (CMA) Valor ...
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,...,,,,,,,,,,Account Executive - Washington DC Our passion ...
4,5,Bill Review Manager,"US, FL, Fort Worth",,,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,...,,,,,,,,,,Bill Review Manager SpotSource Solutions LLC i...


In [8]:
df['y'] = df['fraudulent']                  # Create target variable from the 'fraudulent' column.

In [9]:
df = df[['x', 'y']]                         # Create the dapendant and labeled columns together. 
df

,x,y
0,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,0
3,Account Executive - Washington DC Our passion ...,0
4,Bill Review Manager SpotSource Solutions LLC i...,0
...,...,...
17875,Account Director - Distribution Vend is looki...,0
17876,Payroll Accountant WebLinc is the e-commerce p...,0
17877,Project Cost Control Staff Engineer - Cost Con...,0
17878,Graphic Designer Nemsia Studios is looking fo...,0


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df['x'],
    df['y'],
    test_size=0.2,
    random_state=42
)                                          # Split the dataset into training and testing sets

In [11]:
max_num_words = 10000                      # Convert text to padded sequences for model input
seq_len = 50
embedding_size = 100

tokenizer = Tokenizer(num_words=max_num_words)
tokenizer.fit_on_texts(X_train)

df_train_x = tokenizer.texts_to_sequences(X_train)
df_test_x = tokenizer.texts_to_sequences(X_test)

df_train_x = pad_sequences(df_train_x, maxlen=seq_len)
df_test_x = pad_sequences(df_test_x, maxlen=seq_len)

In [12]:
from keras.layers import Dense, SimpleRNN, Embedding

In [13]:
# model = Sequential()
# model.add(Embedding(input_dim=max_num_words,
#                     output_dim=embedding_size,
#                     input_length=seq_len))

# model.add(Bidirectional(LSTM(128)))
# model.add(Dense(1, activation='sigmoid'))

# model.compile(loss='binary_crossentropy',
#               optimizer=Adam(learning_rate=0.001),
#               metrics=['accuracy'])


In [14]:
model = Sequential()                       # Build and compile the LSTM classification model

model.add(Embedding(input_dim=max_num_words,
                    output_dim=embedding_size))

model.add(Bidirectional(LSTM(128)))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer=Adam(learning_rate=0.001),
              metrics=['accuracy'])

In [15]:
model.fit(df_train_x,                     # Train the model on training data and validate on test data    
          y_train.astype('int32'),
          epochs=5,
          batch_size=32,
          validation_data=(df_test_x, y_test.astype('int32')))

Epoch 1/5
447/447 ━━━━━━━━━━━━━━━━━━━━ 20s 42ms/step - accuracy: 0.9609 - loss: 0.1511 - val_accuracy: 0.9740 - val_loss: 0.1001
Epoch 2/5
447/447 ━━━━━━━━━━━━━━━━━━━━ 20s 46ms/step - accuracy: 0.9823 - loss: 0.0566 - val_accuracy: 0.9762 - val_loss: 0.0946
Epoch 3/5
447/447 ━━━━━━━━━━━━━━━━━━━━ 20s 45ms/step - accuracy: 0.9936 - loss: 0.0221 - val_accuracy: 0.9732 - val_loss: 0.1138
Epoch 4/5
447/447 ━━━━━━━━━━━━━━━━━━━━ 22s 49ms/step - accuracy: 0.9976 - loss: 0.0087 - val_accuracy: 0.9757 - val_loss: 0.1314
Epoch 5/5
447/447 ━━━━━━━━━━━━━━━━━━━━ 23s 50ms/step - accuracy: 0.9990 - loss: 0.0037 - val_accuracy: 0.9776 - val_loss: 0.1805


In [17]:
# Generate predictions and evaluate model performance
y_pred = (model.predict(df_test_x) > 0.5).astype("int32").flatten()

print("Model Evaluation Results")
print("-" * 35)

accuracy = accuracy_score(y_test.astype('int32'), y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test.astype('int32'), y_pred))

print("\nInterpretation:")
print("• The model achieves high overall accuracy, indicating strong performance in detecting real job postings.")
print("• Precision for fake jobs is high, meaning when the model predicts a job as fake, it is usually correct.")
print("• Recall for fake jobs is lower, suggesting some fraudulent postings are still missed.")
print("• Overall, the model performs well but could be improved further for better detection of fraudulent jobs.")

112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step
Model Evaluation Results
-----------------------------------
Overall Accuracy: 97.76%

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       0.90      0.63      0.74       181

    accuracy                           0.98      3576
   macro avg       0.94      0.81      0.86      3576
weighted avg       0.98      0.98      0.98      3576


Interpretation:
• The model achieves high overall accuracy, indicating strong performance in detecting real job postings.
• Precision for fake jobs is high, meaning when the model predicts a job as fake, it is usually correct.
• Recall for fake jobs is lower, suggesting some fraudulent postings are still missed.
• Overall, the model performs well but could be improved further for better detection of fraudulent jobs.
